In [1]:
# ============================================================
# Notebook : 03_causal_discovery.ipynb
# Project  : Causal-Guided Multimodal Sensor Fusion for Elderly Risk Detection
# Purpose  : Temporal causal discovery with PCMCI (ParCorr) on the 8 sensors.
#            PCMCI is run ONCE per household on gap-free segments; per-household
#            significant lagged links are corrected with Benjamini-Hochberg FDR
#            (addresses the multiple-testing concern across segments/lags/pairs).
#            Fold-specific causal graphs are then built by aggregating ONLY the
#            training households of each Group-5-fold split (freq >= 50%),
#            so held-out households never influence the model structure.
#            Outputs: per-fold adjacency tensors, stable-link / sparsity tables,
#            and a grayscale link-frequency heatmap.
# Note     : Anonymous repository for double-blind review.
# ============================================================

from pathlib import Path
import warnings, numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr
import tigramite.data_processing as pp

SEED = 42; np.random.seed(SEED)
FEAT = ['Activity_IR', 'HeartRate', 'BreathRate', 'StressIndex',
        'Temperature', 'Humidity', 'CO2', 'TVOC']
N = len(FEAT); TAU_MAX = 6; ALPHA = 0.05; GAP_MIN = 15
FREQ_THRESH = 0.50; MIN_SEG = 24          # minimal segment length for PCMCI

ROOT     = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"
FIG_DIR  = ROOT / "results" / "figures"; TAB_DIR = ROOT / "results" / "tables"
for d in (PROC_DIR, FIG_DIR, TAB_DIR): d.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="white", context="paper")
mpl.rcParams.update({"figure.dpi":150,"savefig.dpi":600,"savefig.bbox":"tight",
                     "font.family":"DejaVu Sans","axes.unicode_minus":False,"image.cmap":"Greys"})
def savefig(fig, name):
    fig.tight_layout()
    for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

# ── Load table and fold assignment ───────────────────────────
df = pd.read_pickle(PROC_DIR / "df_new.pkl")
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
for c in FEAT: df[c] = pd.to_numeric(df[c], errors='coerce')
df.sort_values(['HH_ID', 'Timestamp'], inplace=True)
folds = pd.read_csv(PROC_DIR / "household_folds.csv")
hh_fold = dict(zip(folds.HH_ID, folds.fold))
N_FOLDS = folds.fold.nunique()

def extract_segments(g):
    g = g.sort_values('Timestamp')
    diff = g['Timestamp'].diff().dt.total_seconds().div(60)
    brk = np.where(diff.values > GAP_MIN)[0]
    bounds = [0, *brk.tolist(), len(g)]
    segs = []
    for a, b in zip(bounds[:-1], bounds[1:]):
        arr = g[FEAT].iloc[a:b].apply(pd.to_numeric, errors='coerce').values
        if len(arr) >= MIN_SEG and not np.isnan(arr).all(axis=0).any():
            segs.append(arr)
    return segs

def bh_fdr(pvals, alpha=ALPHA):
    """Benjamini-Hochberg: return boolean mask of rejected (significant)."""
    p = np.asarray(pvals); m = len(p)
    order = np.argsort(p); ranked = p[order]
    thresh = alpha * (np.arange(1, m + 1) / m)
    passed = ranked <= thresh
    k = np.max(np.where(passed)[0]) + 1 if passed.any() else 0
    mask = np.zeros(m, bool)
    if k > 0: mask[order[:k]] = True
    return mask

# ── Per-household PCMCI (run ONCE per household) ──────────────
pcmci_test = ParCorr(significance='analytic')
hh_links = {}          # hh -> set of (i,j,tau) significant links after BH-FDR
link_rows = []
usable = []
print("Running PCMCI per household (ParCorr, tau_max=6) ...")
for hh, g in df.groupby('HH_ID'):
    segs = extract_segments(g)
    if not segs:
        print(f"  {hh}: no usable segment"); continue
    p_agg = np.ones((N, N, TAU_MAX + 1)); v_agg = np.zeros((N, N, TAU_MAX + 1)); wsum = 0
    for arr in segs:
        # per-segment standardize, fill residual NaNs by column mean (0 after std)
        a = arr.astype(float)
        mu = np.nanmean(a, axis=0); sd = np.nanstd(a, axis=0); sd[sd == 0] = 1.0
        a = (a - mu) / sd
        a = np.nan_to_num(a)
        dataframe = pp.DataFrame(a, var_names=FEAT)
        pc = PCMCI(dataframe=dataframe, cond_ind_test=pcmci_test, verbosity=0)
        res = pc.run_pcmci(tau_min=1, tau_max=TAU_MAX, pc_alpha=ALPHA)
        p_agg = np.minimum(p_agg, res['p_matrix'])      # min p-value across segments
        v_agg += np.abs(res['val_matrix']) * len(a); wsum += len(a)
    v_agg = v_agg / max(wsum, 1)
    # collect lag>=1 p-values, apply BH-FDR across all directed lagged pairs
    idx = [(i, j, t) for i in range(N) for j in range(N) for t in range(1, TAU_MAX + 1)]
    pv = np.array([p_agg[i, j, t] for (i, j, t) in idx])
    sig = bh_fdr(pv, ALPHA)
    links = set()
    for keep, (i, j, t) in zip(sig, idx):
        if keep:
            links.add((i, j, t))
            link_rows.append({'HH_ID': hh, 'Source': FEAT[i], 'Target': FEAT[j],
                              'src_idx': i, 'dst_idx': j, 'Lag': t,
                              'val': round(float(v_agg[i, j, t]), 4),
                              'pval': float(p_agg[i, j, t])})
    hh_links[hh] = links
    usable.append(hh)
    print(f"  {hh}: segments={len(segs):>2}  sig-links(FDR)={len(links)}")

pd.DataFrame(link_rows).to_csv(TAB_DIR / "causal_links_per_household.csv", index=False)
print(f"\nUsable households: {len(usable)}")

# ── Build fold-specific causal adjacency (train households only) ──
sparsity_rows = []
for f in range(N_FOLDS):
    train_hh = [h for h in usable if hh_fold.get(h) != f]
    cnt = np.zeros((N, N, TAU_MAX + 1))
    for h in train_hh:
        for (i, j, t) in hh_links[h]:
            cnt[i, j, t] += 1
    freq = cnt / max(len(train_hh), 1)
    adj = (freq >= FREQ_THRESH).astype(np.float32)     # (N,N,tau+1)
    np.save(PROC_DIR / f"causal_adj_fold{f}.npy", adj)
    mask2d = (adj.sum(axis=2) > 0)
    n_pairs = int(mask2d.sum())
    density = n_pairs / (N * N - N)
    sparsity_rows.append({'fold': f, 'n_train_hh': len(train_hh),
                          'n_link_lag_triplets': int(adj.sum()),
                          'n_directed_pairs': n_pairs,
                          'graph_density': round(density, 3)})
    print(f"fold {f}: train={len(train_hh)}  triplets={int(adj.sum())}  "
          f"directed_pairs={n_pairs}/{N*N-N}  density={density:.2f}")
pd.DataFrame(sparsity_rows).to_csv(TAB_DIR / "causal_graph_sparsity.csv", index=False)

# ── Descriptive stable-link frequency across ALL households ──
cnt_all = np.zeros((N, N))
for h in usable:
    seen = set((i, j) for (i, j, t) in hh_links[h])
    for (i, j) in seen: cnt_all[i, j] += 1
freq_all = cnt_all / max(len(usable), 1)
fig, ax = plt.subplots(figsize=(6.2, 5.2))
sns.heatmap(freq_all, annot=True, fmt=".2f", cmap="Greys", vmin=0, vmax=1,
            xticklabels=FEAT, yticklabels=FEAT, linewidths=0.5, linecolor='white',
            square=True, cbar_kws={'label': 'Link frequency across households'},
            annot_kws={'size': 7}, ax=ax)
ax.set_xlabel('Target sensor'); ax.set_ylabel('Source sensor')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
savefig(fig, "fig06_causal_link_frequency")

# stable links (>=50% across all usable households) as a table
stable = []
for i in range(N):
    for j in range(N):
        if freq_all[i, j] >= FREQ_THRESH:
            stable.append({'Source': FEAT[i], 'Target': FEAT[j],
                           'freq': round(float(freq_all[i, j]), 3)})
pd.DataFrame(stable).sort_values('freq', ascending=False).to_csv(
    TAB_DIR / "stable_causal_links.csv", index=False)

print("\nSaved per-fold adjacencies, sparsity table, link-frequency heatmap.")
print("03_causal_discovery complete.")


Running PCMCI per household (ParCorr, tau_max=6) ...


  A0002: segments=15  sig-links(FDR)=206


  A0003: segments=11  sig-links(FDR)=145


  A0004: segments= 9  sig-links(FDR)=175


  A0005: segments=14  sig-links(FDR)=199
  A0006: no usable segment


...

  A0027: segments=22  sig-links(FDR)=207


  A0028: segments=17  sig-links(FDR)=199


  A0029: segments= 8  sig-links(FDR)=77


  A0030: segments= 8  sig-links(FDR)=112

Usable households: 26
fold 0: train=21  triplets=85  directed_pairs=34/56  density=0.61
fold 1: train=20  triplets=137  directed_pairs=43/56  density=0.77
fold 2: train=21  triplets=99  directed_pairs=37/56  density=0.66
fold 3: train=21  triplets=91  directed_pairs=30/56  density=0.54
fold 4: train=21  triplets=86  directed_pairs=31/56  density=0.55



Saved per-fold adjacencies, sparsity table, link-frequency heatmap.
03_causal_discovery complete.
